# Week 8 (W8-4) - Customer segments from recency, frequency and value
**Owner:** Salim Bello Muhammad (BI Analyst)

**Question.** Phase 3 asks for RFM and segmentation. Do the 500 customers fall into groups that are stable, that predict later behaviour and that differ by region or customer type, or are the segments only labels on a continuous, very regular spread?

**Basis.** Delivered orders only (18,033 of 20,000), revenue is `line_total` before GST, history 2019 to 2024 as in W7-2. For each customer: **recency** (days from last order to the end of the window), **frequency** (delivered orders) and **monetary value** (revenue). Each is scored 1 to 5 by rank (5 is best; for recency 5 is most recent). Five conventional segments follow from the scores: **Champions** (R 4-5 and F+M average 4+), **Loyal** (R 1-3, F+M 4+), **Recent** (R 4-5, F+M below 4), **Slipping** (R 3, F+M below 4) and **Quiet** (R 1-2, F+M below 4).

**What is tested.** (1) Stability: are segments built from 2019-2021 the same as those built from 2022-2024? (2) Prediction: do 2019-2021 segments differ in 2022-2024 orders and revenue? (3) Structure: does k-means find clusters beyond what a structureless cloud of the same shape gives? (4) Profile: do segments differ by region, customer type or industry segment? Seed 42, 5,000 resamples or shuffles, Holm adjustment across tests of one family.

**Limits.** Every customer ordered in 2019 and 2024 (Q13), so no tenure, churn or lifetime-value claim is made. The segments describe the end-of-2024 snapshot only.

In [1]:
import pandas as pd, numpy as np, os
from pathlib import Path
from scipy import stats
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
SURF,INK,INK2,GRID,BLUE,ORANGE,AQUA,MUTED="#fcfcfb","#0b0b0b","#52514e","#e4e3df","#2a78d6","#eb6834","#1baf7a","#b9b8b2"
SEED=42; NB=5000
def find_dir(env_var, marker, rels):
    # 1) environment variable, 2) known layouts from here upward, 3) bounded search of the parent folders
    if os.environ.get(env_var) and (Path(os.environ[env_var]) / marker).exists():
        return Path(os.environ[env_var])
    bases = [Path.cwd(), *Path.cwd().parents]
    for base in bases:
        for rel in rels:
            if (base / rel / marker).exists():
                return base / rel
    skip = {".git", "node_modules", "__pycache__", ".ipynb_checkpoints", ".venv", "venv", "site-packages"}
    for base in bases[:4]:
        found = []
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in skip]
            if len(Path(root).parts) - len(base.parts) >= 7:
                dirs[:] = []
            if marker in files:
                found.append(Path(root))
        if found:
            found.sort(key=lambda p: (p.name != Path(rels[0]).name, len(p.parts), str(p)))
            return found[0]
    return None
MERGED = find_dir("CADETX_MERGED", "sales_orders_full.csv", ["merged", "data/merged", "week-02/reports/week_02/merged", "reports/week_02/merged", "week-02/reports/merged", "reports/merged"])
assert MERGED is not None, "Could not find the Week 2 merged folder. Run from inside the repository, or set CADETX_MERGED to the folder."
print("merged data folder found")
checks=[]
def check(name,expected,actual,ok):
    checks.append(dict(Check_Item=name,Expected=expected,Actual=actual,Status="PASS" if ok else "FAIL")); assert ok,f"{name}: expected {expected}, got {actual}"
def style(ax):
    ax.set_facecolor(SURF); ax.figure.set_facecolor(SURF)
    for s in ["top","right"]: ax.spines[s].set_visible(False)
    for s in ["left","bottom"]: ax.spines[s].set_color(GRID)
    ax.tick_params(colors=INK2); ax.yaxis.grid(True,color=GRID,lw=.8); ax.set_axisbelow(True)
def holm(p):
    p=np.asarray(p,float); o=np.argsort(p); m=len(p); adj=np.empty(m); run=0
    for k,i in enumerate(o): run=max(run,(m-k)*p[i]); adj[i]=min(1,run)
    return adj

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
OUT=Path("w84_outputs"); OUT.mkdir(exist_ok=True); CH=Path("w84_charts"); CH.mkdir(exist_ok=True)
rng=np.random.default_rng(SEED)
S=pd.read_csv(MERGED/"sales_orders_full.csv",low_memory=False,parse_dates=["order_date"],usecols=["so_id","customer_id","order_date","order_status","line_total","customer_region","customer_type","industry_segment"])
# Customer attributes come from the merged sales file itself, so no separate clean folder is needed.
_nu=S.groupby("customer_id")[["customer_region","customer_type","industry_segment"]].nunique().max().max()
assert _nu==1,"customer attributes must be constant within a customer"
CU=S.drop_duplicates("customer_id")[["customer_id","customer_region","customer_type","industry_segment"]].rename(columns={"customer_region":"region"}).reset_index(drop=True)

merged data folder found


## 1. Load and reconcile

In [2]:
dl=S[S.order_status=="Delivered"]
check("delivered orders",18033,dl.so_id.nunique(),dl.so_id.nunique()==18033)
check("delivered revenue before GST (INR B)",22.96,round(dl.line_total.sum()/1e9,2),abs(dl.line_total.sum()/1e9-22.96)<0.005)
check("500 customers in the master and in the orders",500,dl.customer_id.nunique(),dl.customer_id.nunique()==500 and len(CU)==500)
O=dl.groupby(["customer_id","so_id"]).agg(date=("order_date","first"),rev=("line_total","sum")).reset_index()
check("each order belongs to one customer",18033,len(O),len(O)==18033)
def rfm(o,end):
    g=o.groupby("customer_id").agg(F=("so_id","count"),M=("rev","sum"),last=("date","max")); g["R"]=(pd.Timestamp(end)-g["last"]).dt.days; return g[["R","F","M"]]
def score(g):
    s=pd.DataFrame(index=g.index)
    s["r"]=6-pd.qcut(g.R.rank(method="first"),5,labels=False)-1; s["f"]=pd.qcut(g.F.rank(method="first"),5,labels=False)+1; s["m"]=pd.qcut(g.M.rank(method="first"),5,labels=False)+1
    s["r"]=(5-pd.qcut(g.R.rank(method="first"),5,labels=False)).astype(int); s["fm"]=(s.f+s.m)/2
    def seg(x):
        if x.r>=4: return "Champions" if x.fm>=4 else "Recent"
        if x.fm>=4: return "Loyal"
        return "Slipping" if x.r==3 else "Quiet"
    s["segment"]=s.apply(seg,axis=1); return s
FULL=rfm(O,"2024-12-31"); H1=rfm(O[O.date<="2021-12-31"],"2021-12-31"); H2=rfm(O[O.date>="2022-01-01"],"2024-12-31")
check("every customer ordered in both halves",500,int(len(H1.index.intersection(H2.index))),len(H1.index.intersection(H2.index))==500)
check("full-window orders sum to 18,033",18033,int(FULL.F.sum()),int(FULL.F.sum())==18033)
check("half-window orders add up to the full window",int(FULL.F.sum()),int(H1.F.sum()+H2.F.sum()),int(H1.F.sum()+H2.F.sum())==int(FULL.F.sum()))
# plain-loop recomputation for 25 customers
bad=0
for cid in rng.choice(FULL.index.values,25,replace=False):
    sub=O[O.customer_id==cid]; r=(pd.Timestamp("2024-12-31")-sub.date.max()).days
    bad+= int(r!=FULL.loc[cid,"R"] or len(sub)!=FULL.loc[cid,"F"] or abs(sub.rev.sum()-FULL.loc[cid,"M"])>0.01)
check("R, F and M recomputed with a plain filter for 25 customers",0,bad,bad==0)
SC_full=score(FULL); SC1=score(H1); SC2=score(H2)
order=["Champions","Loyal","Recent","Slipping","Quiet"]
check("scores run from 1 to 5 with 100 customers in each fifth",100,int(SC_full.f.value_counts().min()),SC_full.f.value_counts().min()==100 and SC_full.m.value_counts().min()==100 and SC_full.r.value_counts().min()==100)
check("every customer lands in exactly one of the five segments",500,int(SC_full.segment.isin(order).sum()),SC_full.segment.isin(order).all())

## 2. The segments at the end of 2024, with intervals

In [3]:
D=FULL.join(SC_full)
tot=D.M.sum(); rows=[]
for s in order:
    g=D[D.segment==s]; ix=rng.integers(0,500,(NB,500)); seg_m=np.where(D.segment.values==s,D.M.values,0.0)
    share=seg_m[ix].sum(axis=1)/D.M.values[ix].sum(axis=1)*100; size=(D.segment.values==s)[ix].mean(axis=1)*100
    rows.append(dict(segment=s,customers=len(g),size_pct=len(g)/5,size_lo=np.percentile(size,2.5),size_hi=np.percentile(size,97.5),revenue_share_pct=g.M.sum()/tot*100,share_lo=np.percentile(share,2.5),share_hi=np.percentile(share,97.5),median_orders=g.F.median(),median_revenue_m=g.M.median()/1e6,median_days_since_last=g.R.median()))
SG=pd.DataFrame(rows); SG.to_csv(OUT/"w84_segment_summary.csv",index=False)
D[["R","F","M","r","f","m","segment"]].reset_index().to_csv(OUT/"w84_customer_segments.csv",index=False)
check("segment sizes add up to 500",500,int(SG.customers.sum()),SG.customers.sum()==500)
check("segment revenue shares add up to 100%",100.0,round(SG.revenue_share_pct.sum(),6),abs(SG.revenue_share_pct.sum()-100)<1e-6)
print(SG.round(1).to_string(index=False))

  segment  customers  size_pct  size_lo  size_hi  revenue_share_pct  share_lo  share_hi  median_orders  median_revenue_m  median_days_since_last
Champions         70      14.0     11.0     17.2               17.6      14.0      21.3           43.0              56.4                    18.0
    Loyal         86      17.2     14.0     20.6               21.2      17.4      25.1           42.0              55.7                    65.5
   Recent        130      26.0     22.2     29.8               23.0      19.5      26.6           33.0              42.4                    15.0
 Slipping         69      13.8     11.0     17.0               12.4       9.8      15.3           33.0              42.0                    46.0
    Quiet        145      29.0     25.0     33.0               25.7      22.0      29.6           33.0              40.3                   100.0


## 3. Are the segments stable? 2019-2021 against 2022-2024

In [4]:
J=SC1[["segment","r","f","m"]].join(SC2[["segment","r","f","m"]],lsuffix="_1",rsuffix="_2")
ari=adjusted_rand_score(J.segment_1,J.segment_2)
null=np.array([adjusted_rand_score(J.segment_1.values,rng.permutation(J.segment_2.values)) for _ in range(NB)]); p_ari=(null>=ari-1e-12).mean()
stay=(J.segment_1==J.segment_2).mean()*100; pa=J.segment_1.value_counts(normalize=True); pb=J.segment_2.value_counts(normalize=True); exp_stay=sum(pa.get(k,0)*pb.get(k,0) for k in order)*100
print("adjusted Rand index %.3f (chance 0, permutation p %.2f); same segment in both halves: %.1f%% against %.1f%% expected by chance"%(ari,p_ari,stay,exp_stay))
T=pd.crosstab(J.segment_1,J.segment_2).reindex(index=order,columns=order,fill_value=0); T.to_csv(OUT/"w84_transition_matrix.csv")
sp=[]
for k,lab in [("r","recency score"),("f","frequency score"),("m","value score")]:
    rs=stats.spearmanr(J[k+"_1"],J[k+"_2"]).statistic; pp=(np.abs([stats.spearmanr(J[k+"_1"],rng.permutation(J[k+"_2"].values)).statistic for _ in range(NB)])>=abs(rs)-1e-12).mean(); sp.append((lab,rs,pp))
SP=pd.DataFrame(sp,columns=["score","spearman_1_vs_2","perm_p"]); SP["p_holm"]=holm(SP.perm_p.values)
pd.concat([pd.DataFrame([dict(score="segment (adjusted Rand)",spearman_1_vs_2=ari,perm_p=p_ari,p_holm=p_ari)]),SP]).to_csv(OUT/"w84_stability.csv",index=False); print(SP.round(3).to_string(index=False))
check("transition matrix covers 500 customers",500,int(T.values.sum()),T.values.sum()==500)
check("segments are not stable: adjusted Rand index is within chance (permutation p above 0.05)",">0.05",round(p_ari,3),p_ari>0.05)
n=len(J); ct=T.values; sa=sum(x*(x-1)/2 for x in ct.sum(axis=1)); sb=sum(x*(x-1)/2 for x in ct.sum(axis=0)); sij=sum(x*(x-1)/2 for x in ct.flatten()); tt=n*(n-1)/2; ari2=(sij-sa*sb/tt)/(0.5*(sa+sb)-sa*sb/tt)
check("ARI recomputed from the transition matrix by the pair-counting formula",round(ari,6),round(ari2,6),abs(ari-ari2)<1e-9)

adjusted Rand index 0.005 (chance 0, permutation p 0.09); same segment in both halves: 20.4% against 21.8% expected by chance
          score  spearman_1_vs_2  perm_p  p_holm
  recency score           -0.066   0.138   0.415
frequency score            0.025   0.591   1.000
    value score           -0.017   0.714   1.000


## 4. Do 2019-2021 segments predict 2022-2024 behaviour?

In [5]:
P=SC1[["segment"]].join(H2[["F","M"]]); P.columns=["segment","orders_h2","revenue_h2"]
rows=[]
for col,lab in [("orders_h2","orders 2022-2024"),("revenue_h2","revenue 2022-2024")]:
    g=[P[P.segment==s][col].values for s in order if (P.segment==s).sum()>1]; h=stats.kruskal(*g); e=max(0.0,(h.statistic-len(g)+1)/(len(P)-len(g))); rows.append((lab,"Kruskal-Wallis",len(P),h.pvalue,e))
PR=pd.DataFrame(rows,columns=["outcome","test","n","p_value","effect_size_eps2"]); PR["p_holm"]=holm(PR.p_value.values); PR.to_csv(OUT/"w84_prediction.csv",index=False); print(PR.round(4).to_string(index=False))
check("no prediction effect: Holm-adjusted p above 0.05 for both outcomes",">0.05",round(PR.p_holm.min(),3),PR.p_holm.min()>0.05)

          outcome           test   n  p_value  effect_size_eps2  p_holm
 orders 2022-2024 Kruskal-Wallis 500   0.0737            0.0092  0.1473
revenue 2022-2024 Kruskal-Wallis 500   0.6113            0.0000  0.6113


## 5. Is there cluster structure? k-means against a structureless cloud
Customers are described by log recency, log orders and log revenue, standardised. For each k from 2 to 6 the silhouette of k-means is compared with the silhouette of 300 simulated clouds that keep each measure's own distribution and the rank correlation between measures (a Gaussian copula) but contain no clusters by construction. A plain multivariate-normal null is not used because skewed measures raise the silhouette by themselves.

In [6]:
Xr=np.log1p(FULL[["R","F","M"]].values); Xf=(Xr-Xr.mean(0))/Xr.std(0)
U=(pd.DataFrame(Xr).rank().values)/(501); Z=stats.norm.ppf(U); covz=np.corrcoef(Z.T); srt=np.sort(Xf,axis=0)
def sim_cloud():
    z=rng.multivariate_normal(np.zeros(3),covz,500); u=stats.norm.cdf(z); return np.column_stack([np.quantile(srt[:,j],u[:,j]) for j in range(3)])
def sil(X,k): return silhouette_score(X,KMeans(k,n_init=5,random_state=SEED).fit_predict(X))
chk=np.vstack([sim_cloud() for _ in range(20)]); 
check("null clouds keep each measure's mean and spread (within 0.05 SD)","0.05",round(float(np.abs(chk.mean(0)-Xf.mean(0)).max()+np.abs(chk.std(0)-Xf.std(0)).max()),3),np.abs(chk.mean(0)-Xf.mean(0)).max()<0.05 and np.abs(chk.std(0)-Xf.std(0)).max()<0.05)
check("null clouds keep the rank correlation between measures (within 0.1)","0.1",round(float(np.abs(stats.spearmanr(chk).statistic-stats.spearmanr(Xf).statistic).max()),3),np.abs(stats.spearmanr(chk).statistic-stats.spearmanr(Xf).statistic).max()<0.1)
rows=[]
for k in range(2,7):
    o=sil(Xf,k); nl=np.array([sil(sim_cloud(),k) for _ in range(300)]); rows.append(dict(k=k,silhouette=o,null_mean=nl.mean(),null_95th=np.percentile(nl,95),p_value=(nl>=o).mean()))
KM=pd.DataFrame(rows); KM["p_holm"]=holm(KM.p_value.values); KM.to_csv(OUT/"w84_kmeans_structure.csv",index=False); print(KM.round(3).to_string(index=False))
check("five values of k tested",5,len(KM),len(KM)==5)

 k  silhouette  null_mean  null_95th  p_value  p_holm
 2       0.319      0.326      0.348    0.713   1.000
 3       0.335      0.324      0.343    0.177   0.883
 4       0.296      0.291      0.310    0.330   1.000
 5       0.276      0.278      0.301    0.597   1.000
 6       0.259      0.264      0.282    0.643   1.000


## 6. Do segments differ by region, customer type or industry segment?

In [7]:
Pf=D[["segment"]].join(CU.set_index("customer_id")[["region","customer_type","industry_segment"]]); rows=[]
def c2stat(a,b): return stats.chi2_contingency(pd.crosstab(a,b),correction=False)[0]
for col in ["region","customer_type","industry_segment"]:
    tab=pd.crosstab(Pf.segment,Pf[col]); c2,pa,dof,ex=stats.chi2_contingency(tab,correction=False); v=(c2/tab.values.sum()/(min(tab.shape)-1))**0.5
    sg=Pf.segment.values; cv=Pf[col].values; nulls=np.array([c2stat(rng.permutation(sg),cv) for _ in range(2000)]); pp=(1+(nulls>=c2-1e-9).sum())/(1+len(nulls))
    tab.to_csv(OUT/f"w84_profile_{col}.csv"); rows.append((col,"chi-square, permutation p (2,000 shuffles)",500,pp,v,float(ex.min())))
PF=pd.DataFrame(rows,columns=["dimension","test","n","p_value","cramers_v","min_expected_count"]); PF["p_holm"]=holm(PF.p_value.values); PF.to_csv(OUT/"w84_profile_tests.csv",index=False); print(PF.round(4).to_string(index=False))
check("three profile tables cover 500 customers each",1500,int(sum(pd.read_csv(OUT/f"w84_profile_{c}.csv",index_col=0).values.sum() for c in ["region","customer_type","industry_segment"])),sum(pd.read_csv(OUT/f"w84_profile_{c}.csv",index_col=0).values.sum() for c in ["region","customer_type","industry_segment"])==1500)
check("no segment differs by region, type or industry after Holm adjustment","0",int((PF.p_holm<.05).sum()),(PF.p_holm<.05).sum()==0)

       dimension                                       test   n  p_value  cramers_v  min_expected_count  p_holm
          region chi-square, permutation p (2,000 shuffles) 500   0.0970     0.1201               1.518  0.1939
   customer_type chi-square, permutation p (2,000 shuffles) 500   0.5557     0.0862              12.144  0.5557
industry_segment chi-square, permutation p (2,000 shuffles) 500   0.0350     0.1274               9.246  0.1049


## 7. Charts

In [8]:
fig,ax=plt.subplots(figsize=(8.4,4.8)); style(ax); x=np.arange(5); w=.38
ax.bar(x-w/2,SG.size_pct,w,color=BLUE,label="share of customers"); ax.bar(x+w/2,SG.revenue_share_pct,w,color=ORANGE,label="share of revenue")
ax.errorbar(x+w/2,SG.revenue_share_pct,yerr=[SG.revenue_share_pct-SG.share_lo,SG.share_hi-SG.revenue_share_pct],fmt="none",ecolor=INK,capsize=3,lw=1.2)
ax.set_xticks(x); ax.set_xticklabels(SG.segment,color=INK); ax.set_ylabel("%",color=INK2); ax.legend(frameon=False,labelcolor=INK2); ax.set_title("Five segments, each worth about its share of customers",loc="left",color=INK,fontsize=12)
fig.tight_layout(); fig.savefig(CH/"w84_01_segments.png",dpi=150,facecolor=SURF); plt.close(fig)
fig,ax=plt.subplots(figsize=(6.6,5.2)); ax.set_facecolor(SURF); fig.set_facecolor(SURF); Tn=T.div(T.sum(axis=1),axis=0)*100; im=ax.imshow(Tn.values,cmap="Blues",vmin=0,vmax=100)
for i in range(5):
    for j in range(5): ax.text(j,i,f"{Tn.values[i,j]:.0f}%",ha="center",va="center",color=INK,fontsize=9)
ax.set_xticks(range(5)); ax.set_yticks(range(5)); ax.set_xticklabels(order,rotation=30,ha="right",color=INK); ax.set_yticklabels(order,color=INK); ax.set_xlabel("Segment in 2022-2024",color=INK2); ax.set_ylabel("Segment in 2019-2021",color=INK2)
ax.set_title("Where each 2019-2021 segment went (rows add to 100%)",loc="left",color=INK,fontsize=11); fig.tight_layout(); fig.savefig(CH/"w84_02_stability.png",dpi=150,facecolor=SURF); plt.close(fig)
fig,ax=plt.subplots(figsize=(8.4,4.6)); style(ax); ax.plot(KM.k,KM.silhouette,"o-",color=BLUE,label="observed"); ax.plot(KM.k,KM.null_95th,"s--",color=ORANGE,label="structureless cloud, 95th percentile")
ax.set_xlabel("Number of clusters k",color=INK2); ax.set_ylabel("Silhouette",color=INK2); ax.legend(frameon=False,labelcolor=INK2); ax.set_title("k-means finds no more structure than a structureless cloud",loc="left",color=INK,fontsize=12)
fig.tight_layout(); fig.savefig(CH/"w84_03_structure.png",dpi=150,facecolor=SURF); plt.close(fig)
check("three charts written","3",str(len(list(CH.glob('w84_0*.png')))),len(list(CH.glob('w84_0*.png')))==3)

## 8. Output files and validation

In [9]:
outs=sorted(p.name for p in OUT.glob("*.csv")); print(outs)
V=pd.DataFrame(checks); V.insert(0,"Check_ID",range(1,len(V)+1)); V.to_csv("validation_report_week08_w84.csv",index=False)
print(len(V),"checks,",int((V.Status=="PASS").sum()),"pass"); assert (V.Status=="PASS").all()

['w84_customer_segments.csv', 'w84_kmeans_structure.csv', 'w84_prediction.csv', 'w84_profile_customer_type.csv', 'w84_profile_industry_segment.csv', 'w84_profile_region.csv', 'w84_profile_tests.csv', 'w84_segment_summary.csv', 'w84_stability.csv', 'w84_transition_matrix.csv']
22 checks, 22 pass
